# Convert Qiskit & PennyLane circuits to knitweb-lens
Write circuits in your favourite framework, store once, share everywhere via CID.


In [ ]:
!pip install -q knitweb-lens qiskit pennylane

## Qiskit → knitweb-lens

In [ ]:
from qiskit import QuantumCircuit as QC
from knitweb_lens.adapters import from_qiskit

# Build a Qiskit circuit
qc = QC(4, 4, name='qpe_demo')
for i in range(3): qc.h(i)
qc.x(3)
for i in range(3): qc.cp(3.14159 / (2**i), i, 3)
for i in range(3): qc.h(i)
qc.measure_all()

# Convert to lens
lens_circ = from_qiskit(qc, domain='algorithms', tags=['qpe','qiskit'],
                         description='Quantum Phase Estimation from Qiskit')
print(f'CID: {lens_circ.cid}')
print(f'Qubits: {lens_circ.meta.qubits}')
print(lens_circ.qasm[:300])

In [ ]:
# Store and verify
from knitweb_lens.store import Store
store = Store()
cid = store.put(lens_circ)
back = store.get(cid)
print(f'Round-trip OK: {back.meta.name}')

## PennyLane → knitweb-lens

In [ ]:
import pennylane as qml
from knitweb_lens.adapters import from_pennylane

dev = qml.device('default.qubit', wires=4)

@qml.qnode(dev)
def variational_circuit(theta):
    for i in range(4): qml.Hadamard(wires=i)
    for i in range(3): qml.CNOT(wires=[i, i+1])
    for i in range(4): qml.RZ(theta, wires=i)
    return qml.probs(wires=range(4))

# Record the tape
with qml.tape.QuantumTape() as tape:
    for i in range(4): qml.Hadamard(wires=i)
    for i in range(3): qml.CNOT(wires=[i, i+1])
    for i in range(4): qml.RZ(0.5, wires=i)

lens_pl = from_pennylane(tape, name='hw_eff_ansatz', domain='algorithms',
                          tags=['pennylane','ansatz'],
                          description='Hardware-efficient ansatz via PennyLane')
print(f'CID: {lens_pl.cid}')
print(lens_pl.qasm)

## Compare circuit families by CID

In [ ]:
from knitweb_lens import library

lib = library()

# All GHZ variants
ghz_circs = {name: c for name, c in lib.items() if 'ghz' in c.meta.tags}
print('GHZ family:')
for name, c in ghz_circs.items():
    print(f'  {name:<20} q={c.meta.qubits} cid={c.cid[:20]}…')

# CIDs are deterministic — same QASM always gives the same CID
ghz3a = lib['ghz_3'].cid
ghz3b = lib['ghz_3'].cid
assert ghz3a == ghz3b, 'CIDs must be deterministic!'
print(f'\nDeterminism verified: ghz_3 → {ghz3a}')